# Leakage-safe model feature creation

Load the Elo-enriched research matches, derive only seven-day workload and over-six-month layoff history from completed matches on strictly earlier dates, and build the fixed six-feature model datasets.

In [1]:
from pathlib import Path

import pandas as pd

WORKING_DIR = Path.cwd().resolve()
RESEARCH_DIR = WORKING_DIR if WORKING_DIR.name == "research" else WORKING_DIR / "research"
if not RESEARCH_DIR.is_dir():
    raise FileNotFoundError(
        "Run this notebook from the project root or the research directory."
    )

OUTPUT_DIR = RESEARCH_DIR / "data" / "output"
DATA_PATH = OUTPUT_DIR / "matches_2015_onwards_with_elo.csv"

matches = pd.read_csv(DATA_PATH, parse_dates=["Date"], low_memory=False)
input_row_count = len(matches)
input_index = matches.index.copy()

print(f"Loaded {len(matches):,} matches from {DATA_PATH.resolve()}")
print(f"Date range: {matches['Date'].min():%Y-%m-%d} to {matches['Date'].max():%Y-%m-%d}")
matches.head()

Loaded 29,455 matches from /Users/nick/Desktop/tennis live/research/data/output/matches_2015_onwards_with_elo.csv
Date range: 2015-01-05 to 2026-08-14


,ATP,Location,Tournament,Date,Series,Court,Surface,Round,Best of,Winner,...,GlobalMatchLogLoss,GlobalBaseK,SurfaceBaseK,WinnerSurfaceEloPre,LoserSurfaceEloPre,WinnerSurfaceWinProbability,WinnerSurfaceEloPost,LoserSurfaceEloPost,SurfaceMatchLogLoss,SurfaceEffectiveK
0,1,Brisbane,Brisbane International,2015-01-05,ATP250,Outdoor,Hard,1st Round,3.0,Duckworth J.,...,0.693147,32.0,32.0,1500.0,1500.0,0.5,1514.4,1485.6,0.693147,28.8
1,1,Brisbane,Brisbane International,2015-01-05,ATP250,Outdoor,Hard,1st Round,3.0,Kokkinakis T.,...,0.693147,32.0,32.0,1500.0,1500.0,0.5,1514.4,1485.6,0.693147,28.8
2,1,Brisbane,Brisbane International,2015-01-05,ATP250,Outdoor,Hard,1st Round,3.0,Chardy J.,...,0.693147,32.0,32.0,1500.0,1500.0,0.5,1514.4,1485.6,0.693147,28.8
3,1,Brisbane,Brisbane International,2015-01-05,ATP250,Outdoor,Hard,1st Round,3.0,Tomic B.,...,0.693147,32.0,32.0,1500.0,1500.0,0.5,1514.4,1485.6,0.693147,28.8
4,2,Chennai,Chennai Open,2015-01-05,ATP250,Outdoor,Hard,1st Round,3.0,Muller G.,...,0.693147,32.0,32.0,1500.0,1500.0,0.5,1514.4,1485.6,0.693147,28.8


## Pre-match workload and layoff history

Retain only seven-day match workload and over-six-month layoff flags. History contains completed matches from strictly earlier dates; rows sharing a date are assigned before that date updates player state.

In [2]:
from collections import defaultdict, deque

HISTORY_FEATURE_COLUMNS = [
    "WinnerMatchesLast7Days",
    "LoserMatchesLast7Days",
    "WinnerLayoffOver6Months",
    "LoserLayoffOver6Months",
]

feature_values = pd.DataFrame(index=matches.index)
for column in HISTORY_FEATURE_COLUMNS[:2]:
    feature_values[column] = pd.Series(pd.NA, index=matches.index, dtype="Int64")
for column in HISTORY_FEATURE_COLUMNS[2:]:
    feature_values[column] = pd.Series(pd.NA, index=matches.index, dtype="boolean")

context_is_valid = matches[["Date", "Winner", "Loser"]].notna().all(axis=1)
ordered_matches = matches.loc[
    context_is_valid,
    ["Date", "Winner", "Loser", "Comment"],
].copy()
ordered_matches["_row_id"] = ordered_matches.index
ordered_matches = ordered_matches.sort_values(["Date", "_row_id"], kind="stable")

recent_match_dates = defaultdict(deque)
last_match_date = {}

for match_date, day_matches in ordered_matches.groupby("Date", sort=True):
    cutoff_date = match_date - pd.Timedelta(days=7)

    # Assign every row before adding any completed match from this date.
    for row in day_matches.to_dict("records"):
        row_id = row["_row_id"]
        winner = row["Winner"]
        loser = row["Loser"]

        for player in (winner, loser):
            player_dates = recent_match_dates[player]
            while player_dates and player_dates[0] < cutoff_date:
                player_dates.popleft()

        feature_values.at[row_id, "WinnerMatchesLast7Days"] = len(
            recent_match_dates[winner]
        )
        feature_values.at[row_id, "LoserMatchesLast7Days"] = len(
            recent_match_dates[loser]
        )

        winner_last_match = last_match_date.get(winner)
        loser_last_match = last_match_date.get(loser)
        feature_values.at[row_id, "WinnerLayoffOver6Months"] = (
            pd.NA
            if winner_last_match is None
            else bool(match_date > winner_last_match + pd.DateOffset(months=6))
        )
        feature_values.at[row_id, "LoserLayoffOver6Months"] = (
            pd.NA
            if loser_last_match is None
            else bool(match_date > loser_last_match + pd.DateOffset(months=6))
        )

    # Only completed matches become history, and only for later dates.
    completed_day = day_matches.loc[day_matches["Comment"].eq("Completed")]
    for row in completed_day.to_dict("records"):
        winner = row["Winner"]
        loser = row["Loser"]

        recent_match_dates[winner].append(match_date)
        recent_match_dates[loser].append(match_date)
        last_match_date[winner] = match_date
        last_match_date[loser] = match_date

matches = pd.concat(
    [
        matches.drop(columns=HISTORY_FEATURE_COLUMNS, errors="ignore"),
        feature_values,
    ],
    axis=1,
)

print(f"Added {len(HISTORY_FEATURE_COLUMNS)} pre-match history columns.")
matches[["Date", "Winner", "Loser", *HISTORY_FEATURE_COLUMNS]].head()

Added 4 pre-match history columns.


,Date,Winner,Loser,WinnerMatchesLast7Days,LoserMatchesLast7Days,WinnerLayoffOver6Months,LoserLayoffOver6Months
0,2015-01-05,Duckworth J.,Simon G.,0,0,<NA>,<NA>
1,2015-01-05,Kokkinakis T.,Benneteau J.,0,0,<NA>,<NA>
2,2015-01-05,Chardy J.,Golubev A.,0,0,<NA>,<NA>
3,2015-01-05,Tomic B.,Querrey S.,0,0,<NA>,<NA>
4,2015-01-05,Muller G.,Roger-Vasselin E.,0,0,<NA>,<NA>


## History validation

Confirm row preservation, nullable first-appearance layoff flags, nonnegative workloads, and invariant same-day history values.

In [3]:
assert len(matches) == input_row_count, "Feature creation changed the row count."
assert matches.index.equals(input_index), "Feature creation changed the row order."

valid_feature_rows = matches[["Date", "Winner", "Loser"]].notna().all(axis=1)
workload_columns = ["WinnerMatchesLast7Days", "LoserMatchesLast7Days"]
layoff_columns = ["WinnerLayoffOver6Months", "LoserLayoffOver6Months"]

assert matches.loc[valid_feature_rows, workload_columns].notna().all().all()
assert matches.loc[valid_feature_rows, workload_columns].ge(0).all().all()
assert matches[workload_columns].dtypes.astype(str).eq("Int64").all()
assert matches[layoff_columns].dtypes.astype(str).eq("boolean").all()
assert matches[layoff_columns].isna().any().all(), (
    "First appearances should retain unknown layoff flags."
)

history_by_player = pd.concat(
    [
        matches.loc[
            valid_feature_rows,
            [
                "Date",
                "Winner",
                "WinnerMatchesLast7Days",
                "WinnerLayoffOver6Months",
            ],
        ].rename(
            columns={
                "Winner": "Player",
                "WinnerMatchesLast7Days": "MatchesLast7Days",
                "WinnerLayoffOver6Months": "LayoffOver6Months",
            }
        ),
        matches.loc[
            valid_feature_rows,
            [
                "Date",
                "Loser",
                "LoserMatchesLast7Days",
                "LoserLayoffOver6Months",
            ],
        ].rename(
            columns={
                "Loser": "Player",
                "LoserMatchesLast7Days": "MatchesLast7Days",
                "LoserLayoffOver6Months": "LayoffOver6Months",
            }
        ),
    ],
    ignore_index=True,
)
same_day_history_counts = history_by_player.groupby(
    ["Date", "Player"],
    sort=False,
).agg(
    WorkloadValues=("MatchesLast7Days", lambda values: values.nunique(dropna=False)),
    LayoffValues=("LayoffOver6Months", lambda values: values.nunique(dropna=False)),
)
assert same_day_history_counts.le(1).all().all(), (
    "A player's history features changed within a date."
)

print("History feature validation passed, including same-day batching.")
display(
    matches.loc[
        valid_feature_rows,
        ["Date", "Winner", "Loser", *HISTORY_FEATURE_COLUMNS],
    ].head(10)
)
display(
    matches.loc[
        matches[layoff_columns].fillna(False).any(axis=1),
        ["Date", "Winner", "Loser", *layoff_columns, *workload_columns],
    ].head(10)
)

History feature validation passed, including same-day batching.


,Date,Winner,Loser,WinnerMatchesLast7Days,LoserMatchesLast7Days,WinnerLayoffOver6Months,LoserLayoffOver6Months
0,2015-01-05,Duckworth J.,Simon G.,0,0,<NA>,<NA>
1,2015-01-05,Kokkinakis T.,Benneteau J.,0,0,<NA>,<NA>
2,2015-01-05,Chardy J.,Golubev A.,0,0,<NA>,<NA>
3,2015-01-05,Tomic B.,Querrey S.,0,0,<NA>,<NA>
4,2015-01-05,Muller G.,Roger-Vasselin E.,0,0,<NA>,<NA>
5,2015-01-05,Haider-Maurer A.,Granollers M.,0,0,<NA>,<NA>
6,2015-01-05,Coric B.,Haase R.,0,0,<NA>,<NA>
7,2015-01-05,Bolelli S.,Becker B.,0,0,<NA>,<NA>
8,2015-01-05,Dodig I.,Safwat M.,0,0,<NA>,<NA>
9,2015-01-05,Gasquet R.,Andujar P.,0,0,<NA>,<NA>


,Date,Winner,Loser,WinnerLayoffOver6Months,LoserLayoffOver6Months,WinnerMatchesLast7Days,LoserMatchesLast7Days
1772,2015-07-28,Donaldson J.,Devvarman S.,False,True,0,0
2091,2015-08-31,Donskoy E.,Pouille L.,True,False,0,0
2260,2015-09-29,Kukushkin M.,Ramanathan R.,False,True,1,0
2388,2015-10-13,Sock J.,Kubot L.,False,True,2,0
2527,2015-10-27,Paire B.,Gombos N.,False,True,0,0
2632,2016-01-04,Kamke T.,Mitchell B.,True,<NA>,0,0
2639,2016-01-04,Edmund K.,Klizan M.,True,False,0,0
2693,2016-01-07,Ramanathan R.,Kudryavtsev A.,False,True,1,0
2718,2016-01-11,De Bakker T.,Ramos-Vinolas A.,True,False,0,1
2723,2016-01-11,Thompson J.,Klizan M.,True,False,0,1


## Fixed model table

Keep the history-enriched matches in memory and emit only the fixed chronological model splits at the end of the notebook.

## Reproducible orientation and fixed features

Use seed 42 to assign the winner or loser to Player 1 consistently, then retain exactly six ordered model features: global Elo difference, surface Elo difference, rank difference, each player's layoff flag, and seven-day workload difference. Identity, event context, points, separate ratings/ranks/workloads, Elo probabilities, post-match fields, and head-to-head data are excluded.

In [5]:
import numpy as np

RANDOM_SEED = 42
MODEL_FEATURE_COLUMNS = [
    "GlobalEloDiff",
    "SurfaceEloDiff",
    "RankDiff",
    "Player1LayoffOver6Months",
    "Player2LayoffOver6Months",
    "MatchesLast7DaysDiff",
]
MODEL_ODDS_COLUMNS = [
    "B365Player1Odds",
    "B365Player2Odds",
    "PSPlayer1Odds",
    "PSPlayer2Odds",
]
MODEL_COLUMNS = [
    "Date",
    *MODEL_FEATURE_COLUMNS,
    *MODEL_ODDS_COLUMNS,
    "Player1Won",
]
assert len(MODEL_FEATURE_COLUMNS) == 6
assert MODEL_FEATURE_COLUMNS == [
    "GlobalEloDiff",
    "SurfaceEloDiff",
    "RankDiff",
    "Player1LayoffOver6Months",
    "Player2LayoffOver6Months",
    "MatchesLast7DaysDiff",
]

required_model_source_columns = {
    "Date",
    "Comment",
    "Winner",
    "Loser",
    "WRank",
    "LRank",
    "B365W",
    "B365L",
    "PSW",
    "PSL",
    "WinnerGlobalEloPre",
    "LoserGlobalEloPre",
    "WinnerSurfaceEloPre",
    "LoserSurfaceEloPre",
    *HISTORY_FEATURE_COLUMNS,
}
missing_model_source_columns = required_model_source_columns.difference(matches.columns)
assert not missing_model_source_columns, (
    f"Missing required model source columns: {sorted(missing_model_source_columns)}"
)

model_row_is_valid = (
    matches["Comment"].eq("Completed")
    & matches[["Date", "Winner", "Loser"]].notna().all(axis=1)
)
model_source = matches.loc[model_row_is_valid].sort_values("Date", kind="stable").copy()

# A few raw price cells hold "-" instead of a number.
for odds_source_column in ["B365W", "B365L", "PSW", "PSL"]:
    model_source[odds_source_column] = pd.to_numeric(
        model_source[odds_source_column],
        errors="coerce",
    )

rng = np.random.default_rng(RANDOM_SEED)
winner_is_player1 = pd.Series(
    rng.integers(0, 2, size=len(model_source), dtype=np.int8).astype(bool),
    index=model_source.index,
)


def oriented_pair(winner_column: str, loser_column: str) -> tuple[pd.Series, pd.Series]:
    player1_values = model_source[winner_column].where(
        winner_is_player1,
        model_source[loser_column],
    )
    player2_values = model_source[loser_column].where(
        winner_is_player1,
        model_source[winner_column],
    )
    return player1_values, player2_values


player1_global_elo, player2_global_elo = oriented_pair(
    "WinnerGlobalEloPre",
    "LoserGlobalEloPre",
)
player1_surface_elo, player2_surface_elo = oriented_pair(
    "WinnerSurfaceEloPre",
    "LoserSurfaceEloPre",
)
player1_rank, player2_rank = oriented_pair("WRank", "LRank")
player1_workload, player2_workload = oriented_pair(
    "WinnerMatchesLast7Days",
    "LoserMatchesLast7Days",
)
player1_layoff, player2_layoff = oriented_pair(
    "WinnerLayoffOver6Months",
    "LoserLayoffOver6Months",
)
player1_b365_odds, player2_b365_odds = oriented_pair("B365W", "B365L")
player1_ps_odds, player2_ps_odds = oriented_pair("PSW", "PSL")

model_data = pd.DataFrame(index=model_source.index)
model_data["Date"] = model_source["Date"]
model_data["GlobalEloDiff"] = player1_global_elo - player2_global_elo
model_data["SurfaceEloDiff"] = player1_surface_elo - player2_surface_elo
model_data["RankDiff"] = player1_rank - player2_rank
model_data["Player1LayoffOver6Months"] = player1_layoff
model_data["Player2LayoffOver6Months"] = player2_layoff
model_data["MatchesLast7DaysDiff"] = player1_workload - player2_workload
model_data["B365Player1Odds"] = player1_b365_odds
model_data["B365Player2Odds"] = player2_b365_odds
model_data["PSPlayer1Odds"] = player1_ps_odds
model_data["PSPlayer2Odds"] = player2_ps_odds
model_data["Player1Won"] = winner_is_player1.astype("int8")
model_data = model_data.loc[:, MODEL_COLUMNS].reset_index(drop=True)

assert model_data.columns.tolist() == MODEL_COLUMNS
assert model_data[MODEL_FEATURE_COLUMNS].shape[1] == 6
assert (
    model_data[["GlobalEloDiff", "SurfaceEloDiff", "RankDiff", *MODEL_ODDS_COLUMNS]]
    .dtypes.astype(str)
    .eq("float64")
    .all()
), "Elo, rank, and odds columns must be numeric."
assert (
    model_data["RankDiff"].isna().to_numpy()
    == (player1_rank.isna() | player2_rank.isna()).to_numpy()
).all(), "Missing ranks must remain missing in RankDiff."
assert (
    model_data["Player1LayoffOver6Months"].isna().to_numpy()
    == player1_layoff.isna().to_numpy()
).all()
assert (
    model_data["Player2LayoffOver6Months"].isna().to_numpy()
    == player2_layoff.isna().to_numpy()
).all()

print(f"Prepared {len(model_data):,} completed matches for modeling.")
print(f"Player 1 win rate after seeded assignment: {model_data['Player1Won'].mean():.3%}")
model_data.head()

Prepared 28,425 completed matches for modeling.
Player 1 win rate after seeded assignment: 49.295%


,Date,GlobalEloDiff,SurfaceEloDiff,RankDiff,Player1LayoffOver6Months,Player2LayoffOver6Months,MatchesLast7DaysDiff,B365Player1Odds,B365Player2Odds,PSPlayer1Odds,PSPlayer2Odds,Player1Won
0,2015-01-05,0.0,0.0,104.0,<NA>,<NA>,0,4.50,1.18,4.67,1.23,1
1,2015-01-05,0.0,0.0,-124.0,<NA>,<NA>,0,1.44,2.62,1.53,2.67,0
2,2015-01-05,0.0,0.0,-41.0,<NA>,<NA>,0,1.28,3.50,1.34,3.50,1
3,2015-01-05,0.0,0.0,-18.0,<NA>,<NA>,0,2.25,1.57,2.37,1.65,0
4,2015-01-05,0.0,0.0,-70.0,<NA>,<NA>,0,1.66,2.10,1.81,2.10,1


## Chronological model splits

Use 2015–2024 for training, 2025 for validation, and 2026 for the untouched test set. Preserve chronological row order and the identical fixed schema in every split.

In [6]:
TRAIN_START = pd.Timestamp("2015-01-01")
VALIDATION_START = pd.Timestamp("2025-01-01")
TEST_START = pd.Timestamp("2026-01-01")
TEST_END = pd.Timestamp("2027-01-01")

assert model_data["Date"].is_monotonic_increasing
assert model_data["Date"].between(TRAIN_START, TEST_END, inclusive="left").all(), (
    "Model rows must fall within the fixed 2015–2026 split window."
)

train_data = model_data.loc[
    model_data["Date"].between(TRAIN_START, VALIDATION_START, inclusive="left")
].copy()
validation_data = model_data.loc[
    model_data["Date"].between(VALIDATION_START, TEST_START, inclusive="left")
].copy()
test_data = model_data.loc[
    model_data["Date"].between(TEST_START, TEST_END, inclusive="left")
].copy()

splits = {
    "train_2015_2024": train_data,
    "validation_2025": validation_data,
    "test_2026": test_data,
}

assert all(not split.empty for split in splits.values())
assert sum(len(split) for split in splits.values()) == len(model_data)
assert train_data["Date"].max() < validation_data["Date"].min()
assert validation_data["Date"].max() < test_data["Date"].min()

split_summary = []
for split_name, split_data in splits.items():
    assert split_data.columns.tolist() == MODEL_COLUMNS
    assert split_data[MODEL_FEATURE_COLUMNS].shape[1] == 6
    assert set(split_data["Player1Won"].dropna().unique()).issubset({0, 1})
    split_summary.append(
        {
            "Split": split_name,
            "Rows": len(split_data),
            "StartDate": split_data["Date"].min(),
            "EndDate": split_data["Date"].max(),
        }
    )

split_summary = pd.DataFrame(split_summary)
print("Chronological split and exact-schema checks passed.")
split_summary

Chronological split and exact-schema checks passed.


,Split,Rows,StartDate,EndDate
0,train_2015_2024,24112,2015-01-05,2024-12-31
1,validation_2025,2491,2025-01-01,2025-11-16
2,test_2026,1822,2026-01-04,2026-08-14


## Export model datasets

Remove only the three known obsolete split files from `research/data/output`, then write the fixed 2015–2024 train, 2025 validation, and 2026 test CSVs.

In [7]:
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
EXPECTED_OUTPUT_DIR = (RESEARCH_DIR / "data" / "output").resolve()
assert OUTPUT_DIR.resolve() == EXPECTED_OUTPUT_DIR

OBSOLETE_SPLIT_FILENAMES = [
    "model_train_2015_2023.csv",
    "model_validation_2024.csv",
    "model_test_2025_onwards.csv",
]
for filename in OBSOLETE_SPLIT_FILENAMES:
    obsolete_path = (OUTPUT_DIR / filename).resolve()
    assert obsolete_path.parent == EXPECTED_OUTPUT_DIR
    if obsolete_path.exists():
        assert obsolete_path.is_file(), f"Refusing to remove non-file path: {obsolete_path}"
        obsolete_path.unlink()
        print(f"Removed obsolete split: {obsolete_path}")

SPLIT_OUTPUT_PATHS = {
    "train_2015_2024": OUTPUT_DIR / "model_train_2015_2024.csv",
    "validation_2025": OUTPUT_DIR / "model_validation_2025.csv",
    "test_2026": OUTPUT_DIR / "model_test_2026.csv",
}

for split_name, split_data in splits.items():
    output_path = SPLIT_OUTPUT_PATHS[split_name]
    assert output_path.resolve().parent == EXPECTED_OUTPUT_DIR
    assert split_data.columns.tolist() == MODEL_COLUMNS
    assert split_data[MODEL_FEATURE_COLUMNS].shape[1] == 6
    split_data.to_csv(output_path, index=False)
    print(f"Exported {len(split_data):,} rows to {output_path.resolve()}")

Exported 24,112 rows to /Users/nick/Desktop/tennis live/research/data/output/model_train_2015_2024.csv
Exported 2,491 rows to /Users/nick/Desktop/tennis live/research/data/output/model_validation_2025.csv
Exported 1,822 rows to /Users/nick/Desktop/tennis live/research/data/output/model_test_2026.csv
